<a href="https://colab.research.google.com/github/LeonardooAlves/LSE-ST445/blob/main/Week%209/1_st445_week9_lab1_model_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ST445 Week 9 · Lab: evaluating and communicating model performance

**About 2 hours · follows the Week 9 lecture section by section · 22 exercises**

The lecture used a synthetic fraud problem. In this lab you apply the same tools to a **real medical screening problem**, predicting diabetes from routine measurements, and then to a regression and a clustering problem. The final, longest section is about **communication**: turning the numbers into statements, charts and briefings that patients, clinicians and managers can act on.

Each section:

1. explains the technique and why it is used, with links to the documentation;
2. runs a short **demonstration** and interprets its output;
3. sets **two exercises**: 🟢 a simple one that practises the demonstrated tool, and 🔴 a complex one that extends it or asks you to justify a decision.

| Section | Topic | Lecture section |
|---|---|---|
| 0 | Data and set-up | — |
| 1 | A single number can mislead | 01 |
| 2 | Confusion matrices and derived measures | 02 |
| 3 | ROC curves, ROC space and AUC | 03 |
| 4 | Precision–recall curves and the F1 score | 04 |
| 5 | Beyond classification: regression metrics and plots | 05 |
| 6 | Beyond classification: clustering metrics | 05 |
| 7 | **Communicating performance**: natural frequencies, icon arrays, frequency trees, translating metrics, honest charts, uncertainty, and a stakeholder briefing (five parts, ten exercises) | 06 |

**Main documentation:** scikit-learn [User Guide: model evaluation](https://scikit-learn.org/stable/modules/model_evaluation.html) and [Visualizations (the Display API)](https://scikit-learn.org/stable/visualizations.html); Matplotlib [Quick start guide](https://matplotlib.org/stable/users/explain/quick_start.html).

**Before you start:** choose **File → Save a copy in Drive**.

---
## 0. The data: diabetes screening

The **Pima Indians Diabetes** dataset records eight routine measurements for 768 women aged 21 or over, of Pima heritage, near Phoenix, Arizona, and whether they were diagnosed with diabetes within five years (Smith et al., 1988). It is a classic screening problem: about a third of the women developed diabetes, the measurements overlap between the groups, and the two kinds of error have very different consequences for a patient.

* **Source:** originally the US National Institute of Diabetes and Digestive and Kidney Diseases; Smith, J. W. et al. (1988), "Using the ADAP learning algorithm to forecast the onset of diabetes mellitus", *Proceedings of the Symposium on Computer Applications in Medical Care*, 261–265. We load a widely used copy from [GitHub](https://github.com/jbrownlee/Datasets).
* **Columns:** pregnancies; plasma **glucose** (2-hour oral glucose tolerance test, mg/dL); diastolic **blood pressure** (mm Hg); triceps **skin thickness** (mm); 2-hour serum **insulin** (µU/mL); **BMI** (kg/m²); diabetes **pedigree** function (family history score); **age** (years); outcome (1 = diabetes).

### Imports and style

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, recall_score, precision_score, f1_score,
                             matthews_corrcoef, roc_auc_score, average_precision_score, brier_score_loss,
                             confusion_matrix, roc_curve, precision_recall_curve,
                             ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay)

OKABE_ITO = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#56B4E9", "#D55E00", "#F0E442", "#000000"]
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,       # remove the top border of each plot
    "axes.spines.right": False,     # remove the right border of each plot
    "axes.prop_cycle": plt.cycler(color=OKABE_ITO),   # colour-blind-safe colours by default
})
RNG = 445                           # one seed for every random step

### Loading and preprocessing, in one cell

One preprocessing decision matters here: in five columns a value of **0 is physiologically impossible** (nobody has a glucose, blood pressure or BMI of zero), so these zeros are **disguised missing values** (Week 4). We convert them to `NaN`. We do **not** impute them here: imputation is done inside each model's pipeline, so it is fitted on training data only (Section 10 shows why).

In [ ]:
# 🟢 Load and prepare the data: one documented cell
URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
COLUMNS = ["pregnancies", "glucose", "blood_pressure", "skin_thickness", "insulin", "bmi", "pedigree", "age", "diabetes"]

raw = pd.read_csv(URL, header=None, names=COLUMNS)    # the file has no header row
pima = raw.copy()                                      # keep raw unchanged

# 1. Zeros that cannot be real measurements are missing values
IMPOSSIBLE_ZERO = ["glucose", "blood_pressure", "skin_thickness", "insulin", "bmi"]
pima[IMPOSSIBLE_ZERO] = pima[IMPOSSIBLE_ZERO].replace(0, np.nan)

# 2. Features and outcome
X = pima.drop(columns="diabetes")
y = pima["diabetes"]

# 3. One split for the whole lab, stratified so both halves have the same share of diabetes.
#    The test set is used ONLY to evaluate: every choice is made on the training data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=RNG)

assert len(pima) == 768 and y.isin([0, 1]).all()
print("women:", len(pima), "| diabetes:", int(y.sum()), f"({y.mean():.1%})")
print("missing values after recoding:", pima.isna().sum()[pima.isna().sum() > 0].to_dict())
print("training:", X_train.shape, "| test:", X_test.shape, f"| test prevalence {y_test.mean():.1%}")

**What you should see:** 768 women, 268 with diabetes (34.9%). Insulin is missing for 374 women and skin thickness for 227, so a model that cannot handle missing values needs imputation. The test set has 231 women, 81 with diabetes.

### Three models, each a pipeline

A [`pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html) chains preprocessing and a model, so that `fit` fits the imputer and the scaler on the training data only, and `predict_proba` applies them to new data. We use the three models of the lecture's comparison:

* **logistic regression** ([`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)), with median imputation ([`SimpleImputer`](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html)) and standardisation ([`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html));
* **random forest** ([`RandomForestClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html));
* **naive Bayes** ([`GaussianNB`](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html)).

`P` stores each model's predicted probability of diabetes for the 231 test women: the input to every plot that follows.

In [ ]:
models = {
    "logistic regression": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                         LogisticRegression(max_iter=2000)),
    "random forest": make_pipeline(SimpleImputer(strategy="median"),
                                   RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=RNG)),
    "naive Bayes": make_pipeline(SimpleImputer(strategy="median"), GaussianNB()),
}
P = {}
for name, model in models.items():
    model.fit(X_train, y_train)                         # imputer, scaler and model fitted on training data only
    P[name] = model.predict_proba(X_test)[:, 1]         # probability of diabetes for each test woman
print({name: p[:5].round(2) for name, p in P.items()})

---
## 1. A single number can mislead (lecture section 01)

A single metric answers a single question. We compare the three models on several metrics at the default threshold of 0.5, beside a **baseline** that always predicts "no diabetes" ([`DummyClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyClassifier.html) with `strategy="most_frequent"`). A model is only useful if it beats such a baseline on the metric that matters.

Metrics used: [accuracy](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html), [recall](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html) (share of women with diabetes who are flagged), [precision](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html) (share of flagged women who have diabetes), [ROC AUC](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html), [average precision](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.average_precision_score.html) and the [Brier score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.brier_score_loss.html).

In [ ]:
baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)

rows = {"always 'no diabetes'": {"accuracy": accuracy_score(y_test, baseline.predict(X_test)), "recall": 0.0}}
for name, p in P.items():
    y_hat = (p >= 0.5).astype(int)                      # labels at the default threshold
    rows[name] = {"accuracy": accuracy_score(y_test, y_hat), "recall": recall_score(y_test, y_hat),
                  "precision": precision_score(y_test, y_hat), "ROC AUC": roc_auc_score(y_test, p),
                  "average precision": average_precision_score(y_test, p), "Brier score": brier_score_loss(y_test, p)}
metrics_table = pd.DataFrame(rows).T.round(3)
metrics_table

**Interpreting the output:** Doing nothing is 64.9% accurate, because 65% of the test women do not have diabetes. Logistic regression reaches 76.6% accuracy but flags only 53% of the women who have diabetes (recall 0.531). Naive Bayes is less accurate (75.3%) but catches more cases (recall 0.605); logistic regression has the best ROC AUC (0.832), average precision (0.718) and Brier score (0.157). **Which model is 'best' depends on the metric**, and for screening, where a missed case is costly, the high accuracy of a model that misses half the cases is not reassuring.

#### Exercise 1.1 · 🟢 Simple

Accuracy treats both classes as one pool. Compute the **balanced accuracy** ([`balanced_accuracy_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.balanced_accuracy_score.html): the mean of recall and specificity) of the baseline and of the three models at the threshold 0.5. How does the baseline's score change compared with its accuracy?

In [ ]:
# Exercise 1.1: your code here

✍️ *Your interpretation:*

#### Exercise 1.2 · 🔴 Complex

Build a table that **ranks** the three models (1 = best) under each of these metrics: accuracy, balanced accuracy, recall, precision, F1 ([`f1_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html)), Matthews correlation coefficient ([`matthews_corrcoef`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.matthews_corrcoef.html)), ROC AUC, average precision and Brier score (lower is better). Which metrics change the ranking? Write two sentences explaining which metric you would report to a clinician deciding whom to send for a confirmatory test, and why.

In [ ]:
# Exercise 1.2: your code here

✍️ *Your interpretation:*

---
## 2. Confusion matrices and derived measures (lecture section 02)

A confusion matrix counts the four outcomes at one threshold: true negatives, false positives, false negatives and true positives. [`ConfusionMatrixDisplay`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html) draws it; `normalize="true"` divides each **row** by its true-class total (so the diagonal shows recall and specificity), and `normalize="pred"` divides each **column** by its predicted total (so the diagonal shows precision for each prediction). Six **derived measures** come from its four cells (lecture slides 18–24): accuracy, positive predictive value (PPV, precision), true positive rate (TPR, recall), true negative rate (TNR, specificity), false positive rate (FPR = 1 − TNR) and false negative rate (FNR = 1 − TPR). In scikit-learn, rows are the true class and columns the prediction ([User Guide: confusion matrix](https://scikit-learn.org/stable/modules/model_evaluation.html#confusion-matrix)).

In [ ]:
y_hat = (P["logistic regression"] >= 0.5).astype(int)
fig, axs = plt.subplots(1, 3, figsize=(15, 4.3), layout="constrained")
for ax, norm, title, fmt in [(axs[0], None, "Counts", "d"),
                             (axs[1], "true", "Row-normalised (by true class)", ".2f"),
                             (axs[2], "pred", "Column-normalised (by prediction)", ".2f")]:
    ConfusionMatrixDisplay.from_predictions(y_test, y_hat, normalize=norm, values_format=fmt,
                                            display_labels=["no diabetes", "diabetes"],
                                            cmap="Blues", colorbar=False, ax=ax)
    ax.set_title(title)
plt.show()
print("TN, FP, FN, TP =", confusion_matrix(y_test, y_hat).ravel())

**Interpreting the output:** Of 231 test women, 134 without diabetes are correctly cleared, 16 are flagged unnecessarily, 38 with diabetes are missed and 43 are correctly flagged. The **row-normalised** matrix shows the asymmetry: 89% of women without diabetes are correctly cleared, but only 53% of women with diabetes are flagged; 47% are missed. The **column-normalised** matrix shows that 73% of flagged women do have diabetes (precision), and 78% of cleared women are truly clear. The default threshold favours avoiding false alarms at the price of many missed cases.

#### Exercise 2.1 · 🟢 Simple

From the four counts printed above, compute **by hand** the six derived measures of the lecture: accuracy, PPV (precision), TPR (recall), TNR (specificity), FPR and FNR. Check that TPR + FNR = 1 and TNR + FPR = 1, and that your values match `recall_score`, `precision_score` and `accuracy_score`.

In [ ]:
# Exercise 2.1: your code here

✍️ *Your interpretation:*

#### Exercise 2.2 · 🔴 Complex

Draw the **row-normalised** confusion matrix of logistic regression at three thresholds, **0.3, 0.5 and 0.7**, side by side. Describe how the two kinds of error move as the threshold changes, and say which threshold you would recommend for a screening programme where a confirmatory test follows a positive result.

In [ ]:
# Exercise 2.2: your code here

✍️ *Your interpretation:*

---
## 3. ROC curves, ROC space and AUC (lecture section 03)

An ROC curve plots the true positive rate against the false positive rate at **every** threshold ([`roc_curve`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_curve.html)). The area under it ([`roc_auc_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html)) is the probability that a randomly chosen woman with diabetes gets a higher score than a randomly chosen woman without (Hanley and McNeil, 1982). [`RocCurveDisplay`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.RocCurveDisplay.html) draws the curve; `plot_chance_level=True` adds the diagonal of random ranking ([User Guide: ROC](https://scikit-learn.org/stable/modules/model_evaluation.html#receiver-operating-characteristic-roc)).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5.5))
for i, (name, model) in enumerate(models.items()):
    RocCurveDisplay.from_estimator(model, X_test, y_test, name=name, ax=ax,
                                   plot_chance_level=(i == len(models) - 1))   # draw the diagonal once
ax.set(title="ROC curves, diabetes test set (n = 231)")
ax.set_aspect("equal")
plt.show()

**Interpreting the output:** The three curves are close and cross several times: logistic regression has the largest area (AUC 0.832), random forest 0.821 and naive Bayes 0.798, but no model is above the others at every threshold. With only 81 women with diabetes in the test set, each curve moves in steps of 1/81 in TPR, and differences of 0.01 in AUC are well within sampling noise (Exercise 7.6 measures this).

#### Exercise 3.1 · 🟢 Simple

**ROC space** (lecture slide 34). At the default threshold 0.5, each model is a single **point** (FPR, TPR). Compute the three points and plot them on empty ROC axes with the diagonal, labelling each point. Using the lecture's rule (A is better than B if A is to the left of and higher than B), does any model **dominate** another at this threshold?

In [ ]:
# Exercise 3.1: your code here

✍️ *Your interpretation:*

#### Exercise 3.2 · 🔴 Complex

Build the ROC curve of logistic regression **by hand**: for each distinct predicted probability t (from highest to lowest), compute TPR(t) and FPR(t) with NumPy, plot the curve, and compute its area with the trapezium rule (`np.trapezoid`). Then verify the ranking interpretation: compute the share of (diabetes, no diabetes) pairs in which the woman with diabetes has the higher score (ties count half). All three numbers should agree with `roc_auc_score`.

In [ ]:
# Exercise 3.2: your code here

✍️ *Your interpretation:*

---
## 4. Precision–recall curves and the F1 score (lecture section 04)

A PR curve plots precision against recall at every threshold ([`precision_recall_curve`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_recall_curve.html)); [`average_precision_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.average_precision_score.html) summarises it as a step-wise area. The chance level is a horizontal line at the **prevalence**, not a diagonal. [`PrecisionRecallDisplay`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.PrecisionRecallDisplay.html) draws it ([User Guide: precision–recall](https://scikit-learn.org/stable/modules/model_evaluation.html#precision-recall-f-measure-metrics)).

The **F1 score** ([`f1_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html)) is the harmonic mean of precision and recall, F1 = 2PR / (P + R), at **one** threshold; average precision summarises **all** thresholds (lecture slide 48: AUC-PR against F1).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5.5))
for i, (name, model) in enumerate(models.items()):
    PrecisionRecallDisplay.from_estimator(model, X_test, y_test, name=name, ax=ax,
                                          plot_chance_level=(i == len(models) - 1))   # the prevalence line, once
ax.set(title=f"Precision–recall curves (prevalence {y_test.mean():.1%})", ylim=(0, 1.02))
plt.show()

**Interpreting the output:** At low recall all three models are precise (the women with the highest scores almost all have diabetes); precision falls as the threshold is lowered to catch more cases. Average precision is 0.718 for logistic regression, 0.697 for random forest and 0.671 for naive Bayes, against a chance level of 0.351, the test prevalence. To catch 80% of cases, about half of the flagged women would not have diabetes.

#### Exercise 4.1 · 🟢 Simple

Compute each model's **F1 score at the default threshold 0.5**. Then, from the arrays returned by `precision_recall_curve`, find the threshold that **maximises F1** for logistic regression, and report precision, recall and F1 there. How far is the F1-optimal threshold from 0.5?

In [ ]:
# Exercise 4.1: your code here

✍️ *Your interpretation:*

#### Exercise 4.2 · 🔴 Complex

Show that **AP depends on prevalence but ROC AUC does not**. Using logistic regression's test scores, draw 200 random subsamples in which diabetes makes up only **10%** of the women (keep all women without diabetes, sample the right number with diabetes). For each, compute ROC AUC and AP. Compare the mean and standard deviation with the full test set, and plot the two distributions. Explain the result in two sentences.

In [ ]:
# Exercise 4.2: your code here

✍️ *Your interpretation:*

---
## 5. Beyond classification: regression (lecture section 05)

A regression model predicts a number, so there is no confusion matrix; the same logic applies, **plots first, metrics second**. The lecture's metrics (slides 52–56): mean absolute error ([`MAE`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_error.html), in the target's units, robust to outliers), mean squared error and its root ([`MSE`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html), RMSE, which penalise large errors), [`R²`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html) (share of variance explained), [`explained variance`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.explained_variance_score.html) and [`max error`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.max_error.html) (the single worst prediction). [`PredictionErrorDisplay`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.PredictionErrorDisplay.html) draws actual against predicted and residuals against predicted.

**The data:** scikit-learn's bundled diabetes **progression** dataset ([`load_diabetes`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html)): a disease-progression score one year after baseline (25 to 346) for 442 patients, with 10 standardised baseline measurements. As on lecture slide 56, we compare a linear model with a degree-2 polynomial ([`PolynomialFeatures`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html)) using **BMI** alone, so every plot can be read directly.

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import (PredictionErrorDisplay, mean_absolute_error, mean_squared_error,
                             r2_score, explained_variance_score, max_error)

Xr, yr = load_diabetes(return_X_y=True, as_frame=True)
Xb = Xr[["bmi"]]                                           # one feature: BMI (standardised)
Xb_tr, Xb_te, yr_tr, yr_te = train_test_split(Xb, yr, test_size=0.3, random_state=RNG)

linear = LinearRegression().fit(Xb_tr, yr_tr)
quadratic = make_pipeline(PolynomialFeatures(degree=2), LinearRegression()).fit(Xb_tr, yr_tr)

def regression_metrics(y_true, y_pred):
    """The six regression metrics of the lecture, as a dictionary."""
    mse = mean_squared_error(y_true, y_pred)
    return {"MAE": mean_absolute_error(y_true, y_pred), "MSE": mse, "RMSE": mse ** 0.5,
            "R²": r2_score(y_true, y_pred), "explained variance": explained_variance_score(y_true, y_pred),
            "max error": max_error(y_true, y_pred)}

fig, axs = plt.subplots(1, 2, figsize=(12, 4.5), layout="constrained")
PredictionErrorDisplay.from_estimator(linear, Xb_te, yr_te, kind="actual_vs_predicted", ax=axs[0], scatter_kwargs={"alpha": 0.6})
PredictionErrorDisplay.from_estimator(linear, Xb_te, yr_te, kind="residual_vs_predicted", ax=axs[1], scatter_kwargs={"alpha": 0.6})
axs[0].set_title("Linear model on BMI: actual vs predicted"); axs[1].set_title("Residuals vs predicted")
plt.show()
pd.DataFrame({"linear": regression_metrics(yr_te, linear.predict(Xb_te))}).round(3)

**Interpreting the output:** BMI alone captures a clear upward trend, but the points scatter widely around the diagonal: the typical error (MAE) is about 53 points on a 25–346 scale, and R² is only 0.24, so BMI explains about a quarter of the variation in progression. The residuals show no strong curve, but their spread is wide everywhere: the model is not wrong in shape so much as missing information. The max error shows the worst single patient is predicted about 164 points off.

#### Exercise 5.1 · 🟢 Simple

Fit the **degree-2 polynomial** model already defined (`quadratic`) and put its six metrics beside the linear model's in one table. Does the curve improve the fit? Draw both models' residual plots side by side to support your answer.

In [ ]:
# Exercise 5.1: your code here

✍️ *Your interpretation:*

#### Exercise 5.2 · 🔴 Complex

**Outliers** (lecture slide 55). Copy the test targets and corrupt **5 of them** by multiplying them by 3 (as a data-entry error might). Recompute the six metrics of the linear model on the corrupted targets and show, as a table of **percentage changes**, which metrics are most and least sensitive. Draw the actual-vs-predicted plot with the corrupted points highlighted. Which metric would you report to a clinician if the data might contain such errors, and why?

In [ ]:
# Exercise 5.2: your code here

✍️ *Your interpretation:*

---
## 6. Beyond classification: clustering (lecture section 05)

Clustering is usually unsupervised, so evaluation depends on whether labels exist (lecture slide 57). **Internal** metrics use only the data: the **silhouette** of a point is $s = (b − a) / \max(a, b)$, where $a$ is its mean distance to its own cluster and $b$ to the nearest other cluster; it lies between −1 and 1 ([`silhouette_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html), [`silhouette_samples`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_samples.html)). **External** metrics compare clusters with known labels: the **adjusted Rand index** ([`adjusted_rand_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_rand_score.html)) is 0 for random agreement and 1 for perfect agreement.

We cluster the Pima women's eight measurements (imputed and standardised) with [`KMeans`](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html) and ask: do the measurements contain natural groups?

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples, adjusted_rand_score

Zp = make_pipeline(SimpleImputer(strategy="median"), StandardScaler()).fit_transform(X)   # all 768 women
labels2 = KMeans(n_clusters=2, n_init=10, random_state=RNG).fit_predict(Zp)
sv = silhouette_samples(Zp, labels2)

fig, ax = plt.subplots(figsize=(6.5, 4.2))
y0 = 0
for j in range(2):
    s = np.sort(sv[labels2 == j])
    ax.fill_betweenx(np.arange(y0, y0 + len(s)), 0, s, alpha=0.8, label=f"cluster {j} (n = {len(s)})")
    y0 += len(s) + 10
ax.axvline(sv.mean(), ls="--", color="grey")
ax.set(title=f"Silhouette plot, k = 2 (mean {sv.mean():.2f})", xlabel="silhouette value", yticks=[]); ax.legend()
plt.show()
print(f"mean silhouette {sv.mean():.3f} | share negative {np.mean(sv < 0):.1%}")

**Interpreting the output:** The mean silhouette is low (about 0.2): many women sit near the boundary between the two clusters, and about one in seven has a negative silhouette (closer, on average, to the other cluster than to its own). A good clustering has thick bands extending well past the mean, with few negative values (lecture slide 59); this is a weak structure. K-means always returns k clusters, so the plot, not the existence of clusters, is the evidence.

#### Exercise 6.1 · 🟢 Simple

Compute the mean silhouette for **k = 2 to 7** and plot it against k. Which k has the highest silhouette, and is that value high enough to claim that the women fall into natural groups?

In [ ]:
# Exercise 6.1: your code here

✍️ *Your interpretation:*

#### Exercise 6.2 · 🔴 Complex

Now use the **external** view. Compare the k = 2 clusters with diabetes status using the **adjusted Rand index** and a cross-tabulation (`pd.crosstab(..., normalize="index")`). Then describe each cluster by the median of its eight measurements. Write a three-sentence summary for a clinician: what the clusters are, how they relate to diabetes, and why the clusters should not be presented as 'patient types'.

In [ ]:
# Exercise 6.2: your code here

✍️ *Your interpretation:*

---
## 7. Communicating performance (lecture section 6)

This is the longest section, because communication is where evaluation becomes useful. It has five parts, each with a simple and a complex exercise:

| Part | Skill | Lecture slide |
|---|---|---|
| 7A | Natural frequencies and outcomes per 1,000 | 61 |
| 7B | Icon arrays and frequency trees | 62 |
| 7C | Translating metrics into decisions | 63 |
| 7D | Avoiding misleading presentations, and showing uncertainty | 64 |
| 7E | Stakeholder challenges: answer a hard question with one clear visual | 61–65 |

All the parts use **logistic regression** (the best-ranking and best-calibrated model of the earlier sections) and two operating points: the default threshold **0.5**, and a **screening threshold of 0.3**, which Exercise 2.2 showed catches far more cases. The helper below turns any threshold into **outcomes per 1,000 women**, scaling the test-set rates (Gigerenzer and Hoffrage, 1995).

In [ ]:
p_lr = P["logistic regression"]
y_arr = y_test.to_numpy()

def per_1000(t, p=p_lr, yt=y_arr):
    """Outcomes per 1,000 women screened at threshold t, scaled from the test set."""
    flagged = p >= t
    scale = 1000 / len(yt)
    tp = scale * np.sum(flagged & (yt == 1)); fn = scale * np.sum(~flagged & (yt == 1))
    fp = scale * np.sum(flagged & (yt == 0)); tn = scale * np.sum(~flagged & (yt == 0))
    return {"with diabetes": round(tp + fn), "found": round(tp), "missed": round(fn),
            "false alarms": round(fp), "correctly cleared": round(tn), "positive results": round(tp + fp)}

outcomes = pd.DataFrame({"threshold 0.5": per_1000(0.5), "threshold 0.3": per_1000(0.3)})
outcomes

**Interpreting the output:** Out of every 1,000 women, about 351 have diabetes. At threshold 0.5 the screen finds about 186 and misses about 165, with about 69 false alarms. At 0.3 it finds about 277 and misses about 74, at the price of about 203 false alarms. The table states the trade-off in counts that anyone can compare: about 91 more cases found for about 134 more false alarms.

### 7A · Natural frequencies

#### Exercise 7.1 · 🟢 Simple

Write a function `leaflet(t)` that returns **one plain-English sentence** for a patient leaflet, using natural frequencies with 1,000 as the reference class throughout, for example: *'Of every 1,000 women screened, about X have diabetes. The test finds Y of them and misses Z. Another W women without diabetes are told to take a further test.'* Print it for both thresholds.

In [ ]:
# Exercise 7.1: your code here

✍️ *Your interpretation:*

#### Exercise 7.2 · 🔴 Complex

**The positive predictive value question.** A woman is told her screening result is positive. Using natural frequencies, write the answer to her question *'What is the chance I really have diabetes?'* at both thresholds (positives found among all positive results). Then show that the answer **depends on the base rate**: recompute it for a population where only **10%** of women have diabetes, keeping the test's TPR and FPR at threshold 0.3, and write the answer as a natural-frequency sentence. Why might a GP in a low-risk practice be surprised?

In [ ]:
# Exercise 7.2: your code here

✍️ *Your interpretation:*

### 7B · Icon arrays and frequency trees

#### Exercise 7.3 · 🟢 Simple

Draw an **icon array** of 1,000 women at threshold 0.3: one square per woman, coloured by outcome (found, missed, false alarm, correctly cleared), grouped by outcome so that the counts can be seen, with a legend giving the counts and a title that states the finding (Galesic et al., 2009). Use the colour-blind-safe palette.

In [ ]:
# Exercise 7.3: your code here

✍️ *Your interpretation:*

#### Exercise 7.4 · 🔴 Complex

Draw a **frequency tree** (natural-frequency tree): a top box with 1,000 women, splitting into 'with diabetes' and 'without diabetes', each splitting into 'positive result' and 'negative result', with the count in every box and arrows between them. Draw it for threshold 0.3 using only Matplotlib (boxes with `ax.text(..., bbox=...)` and arrows with `ax.annotate`). Below the tree, add one sentence answering *'If my result is positive, what is the chance I have diabetes?'* with the two numbers that the tree makes visible. Explain why trees are recommended for Bayesian reasoning (Gigerenzer and Hoffrage, 1995).

In [ ]:
# Exercise 7.4: your code here

✍️ *Your interpretation:*

### 7C · Translating metrics into decisions

#### Exercise 7.5 · 🟢 Simple

Write a function `translate(t)` that prints a **two-column translation table** (lecture slide 63) for logistic regression at threshold t: the technical metric and value on the left (recall, precision, FPR, FNR, accuracy), and a plain-language sentence per 1,000 women on the right. Show it for threshold 0.3.

In [ ]:
# Exercise 7.5: your code here

✍️ *Your interpretation:*

#### Exercise 7.6 · 🔴 Complex

**A one-page briefing figure** for the head of a screening service, who has five minutes. Build a single Matplotlib figure (use `plt.subplot_mosaic` or `GridSpec`) with: (1) a headline title that states the recommendation; (2) a grouped bar chart comparing, per 1,000 women, cases found, cases missed and false alarms at thresholds 0.5 and 0.3, with values written on the bars; (3) a short text panel with the plain-language trade-off and one caveat; (4) a small panel with the 95% bootstrap interval for 'cases found per 1,000' at each threshold, so that the uncertainty is visible. Keep AUC off the main figure (lecture slide 63).

In [ ]:
# Exercise 7.6: your code here

✍️ *Your interpretation:*

### 7D · Avoiding misleading presentations, and showing uncertainty

The cell below draws a chart in the style the lecture warns against (slide 64): two models' AUCs as bars on a **truncated axis**, with no interval. Run it, then fix it in the exercises.

In [ ]:
aucs = {name: roc_auc_score(y_test, p) for name, p in P.items()}
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(list(aucs), list(aucs.values()), color=OKABE_ITO[:3])
ax.set_ylim(0.79, 0.835)                                        # truncated axis: the misleading choice
ax.set(title="Logistic regression is far better!", ylabel="AUC")
plt.show()

**Interpreting the output:** With the y-axis starting at 0.79, a difference of 0.034 in AUC between the best and worst model looks like a factor of several, and the title claims a large advantage. No interval is shown, so the reader cannot tell that the differences are within sampling noise.

#### Exercise 7.7 · 🟢 Simple

Redraw the chart honestly: a **dot plot** (position, not bar length) of the three AUCs on an axis from 0.5 (chance) to 1, with a title that states the finding fairly. Explain in one sentence why a dot plot does not need a zero baseline while a bar chart does.

In [ ]:
# Exercise 7.7: your code here

✍️ *Your interpretation:*

#### Exercise 7.8 · 🔴 Complex

Add **uncertainty**: compute a 95% paired bootstrap interval for each model's AUC (1,000 resamples of the 231 test women, all three models scored on each resample) and draw the dot plot with intervals. Then compute the interval for the **difference** between logistic regression and naive Bayes. Rewrite the misleading title as a statement a newspaper could print without being wrong.

In [ ]:
# Exercise 7.8: your code here

✍️ *Your interpretation:*

### 7E · Stakeholder challenges: one hard question, one clear visual

#### Exercise 7.9 · 🟢 Simple

**Challenge card: the GP.** *'My practice has 600 women aged 21–45 eligible for screening. If we screen them all at threshold 0.3, how many referrals will I have to make, and how many of those women will actually have diabetes?'* Answer with a **stacked horizontal bar** of the 600 women split into the four outcomes (labelled with counts), and a one-sentence answer in the title. Use the test-set rates for women aged 21–45 only, and state that assumption in a caption.

In [ ]:
# Exercise 7.9: your code here

✍️ *Your interpretation:*

#### Exercise 7.10 · 🔴 Complex

**Challenge card: the board.** *'If we roll this screen out to 10,000 women next year, how many cases will we find, and how sure are you?'* Design a **creative but honest visual of uncertainty**: a *hypothetical outcomes* display (Hullman, Resnick and Adar, 2015) of six small icon-style panels (or bar panels), each showing one plausible year drawn from bootstrap resamples of the test set at threshold 0.3, plus a summary line with the 95% interval for cases found and false alarms per 10,000. Write the two-sentence answer you would give the board, and one sentence on why showing several plausible outcomes can communicate uncertainty better than a single interval.

In [ ]:
# Exercise 7.10: your code here

✍️ *Your interpretation:*

---
## Summary

| Question | Tool | Section |
|---|---|---|
| Is the model better than doing nothing? | a baseline (`DummyClassifier`), several metrics | 1 |
| What errors does it make at one threshold? | confusion matrix; accuracy, PPV, TPR, TNR, FPR, FNR | 2 |
| How well does it rank cases; which point in ROC space? | ROC curve, AUC, ROC-space points | 3 |
| How pure are its alerts at this prevalence? | PR curve, average precision, F1 | 4 |
| How good is a regression? | actual-vs-predicted and residual plots; MAE, RMSE, R², max error | 5 |
| Are there natural groups? | silhouette (internal), adjusted Rand index (external) | 6 |
| How do we explain it to people? | natural frequencies, icon arrays, frequency trees, translation tables, honest charts with intervals, briefing figures | 7 |

### References

Galesic, M., Garcia-Retamero, R. and Gigerenzer, G. (2009), "Using icon arrays to communicate medical risks", *Health Psychology*, 28(2), 210–216 · Gigerenzer, G. and Hoffrage, U. (1995), "How to improve Bayesian reasoning without instruction: frequency formats", *Psychological Review*, 102(4), 684–704 · Hanley, J. A. and McNeil, B. J. (1982), *Radiology*, 143(1), 29–36 · Hubert, L. and Arabie, P. (1985), "Comparing partitions", *Journal of Classification*, 2, 193–218 · Hullman, J., Resnick, P. and Adar, E. (2015), "Hypothetical outcome plots outperform error bars and violin plots for inferences about reliability of variable ordering", *PLOS ONE*, 10(11), e0142444 · Kaufman, L. and Rousseeuw, P. J. (1990), *Finding Groups in Data*, Wiley · Pedregosa, F. et al. (2011), "Scikit-learn", *JMLR*, 12, 2825–2830 · Rousseeuw, P. J. (1987), "Silhouettes", *Journal of Computational and Applied Mathematics*, 20, 53–65 · Saito, T. and Rehmsmeier, M. (2015), *PLOS ONE*, 10(3), e0118432 · Smith, J. W. et al. (1988), *Proc. Symposium on Computer Applications in Medical Care*, 261–265 · Spiegelhalter, D. (2017), "Risk and uncertainty communication", *Annual Review of Statistics and Its Application*, 4, 31–60.